# Paso 2 · Entender Tavily y evaluar su eficacia

Preguntas de este paso:

1. ¿Qué ve el LLM cuando hacemos `bind_tools()`? ¿Qué parámetros puede usar?
2. ¿Qué devuelve Tavily? ¿Qué estructura tiene el mensaje?
3. ¿Cómo se generan los *snippets*? ¿Son realmente la parte relevante de la página?
4. ¿Cómo se genera el *score*? ¿Concuerda con la relevancia real?
5. ¿El modelo usa de verdad los resultados o se inventa cosas?

> Código en `src/espadas_langgraph_websearchagent/tavily_analysis.py` y `judge.py`.

## 0. Configuración

In [ ]:
# Cargamos las claves de API desde el fichero .env (ANTHROPIC_API_KEY y TAVILY_API_KEY).
# Si no están, las pedimos por teclado.
import getpass
import os

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key:\n")

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

In [ ]:
from espadas_langgraph_websearchagent import searchagent
from espadas_langgraph_websearchagent import tavily_analysis as ta
from espadas_langgraph_websearchagent.tokens import count_text_tokens, count_tool_definition_tokens

model = searchagent.create_model()
tool = searchagent.create_default_search_tool()

## 1. ¿Qué ve el LLM? La "API" de la tool después de `bind_tools()`

`bind_tools()` convierte cada tool a un JSON con tres partes:
- `name`: `tavily_search`
- `description`: cuándo usar la tool
- `input_schema`: los parámetros que el LLM puede rellenar, cada uno con su tipo y su descripción

Este JSON se envía **en todas las llamadas al LLM**, se busque o no. Además, Anthropic añade un *system prompt* oculto de **496 tokens** (Claude Haiku 4.5) cada vez que hay tools. Estos son los ~2.000 tokens que aparecen en el primer `input_tokens`.

In [ ]:
# El JSON exacto que recibe el LLM
ta.print_tool_schema(tool)

In [ ]:
# Resumen de los parámetros que puede rellenar el LLM
ta.print_tool_parameters(tool)

tool_tokens = count_tool_definition_tokens(model, [tool])
print()
print(f"Tokens de tener la tool enchufada: {tool_tokens:,}")
print(f"  - system prompt oculto de Anthropic (Haiku 4.5): 496")
print(f"  - JSON de la tool de Tavily: ≈ {tool_tokens - 496:,}")

### Qué puede decidir el LLM y qué decidimos nosotros

| Lo rellena el LLM (está en el schema) | Lo fijamos al crear la tool (el LLM no lo ve) |
|---|---|
| `query` (obligatorio) | `max_results` |
| `include_domains`, `exclude_domains` | `include_answer` |
| `search_depth` (`basic` / `advanced` / `fast` / `ultra-fast`) | `include_raw_content` |
| `include_images` | `country` |
| `time_range`, `start_date`, `end_date` | `include_favicon`, `include_usage` |
| `topic` (`general` / `news` / `finance`) | `chunks_per_source` (solo con la API directa) |

Leyendo el código de `langchain_tavily` vemos que, **si fijamos un parámetro al crear la tool, el valor del LLM se ignora** (`self.search_depth if self.search_depth else search_depth`).

En el ejemplo de la Honda Rebel del Paso 1, el LLM eligió por su cuenta `search_depth="advanced"`, que cuesta **2 créditos** de Tavily en vez de 1.

**Observación:** casi todo el JSON son descripciones larguísimas de parámetros que el agente casi nunca usa (fechas, dominios, imágenes...). Las pagamos en cada llamada.

## 2. ¿Qué devuelve Tavily?

Llamamos a la tool directamente (sin LLM) y miramos la estructura.

In [ ]:
response = tool.invoke({"query": "última versión de Python"})
ta.describe_response(response)

### Estructura de la respuesta

| Campo | Qué es |
|---|---|
| `query` | la query que se buscó |
| `follow_up_questions` | (vacío) preguntas sugeridas |
| `answer` | respuesta generada por un LLM de Tavily (solo con `include_answer`) |
| `images` | imágenes (solo con `include_images`) |
| `results` | **lista de resultados**, ordenados por relevancia |
| `results[i].title` / `url` | título y dirección de la página |
| `results[i].content` | el **snippet**: los trozos de la página más relacionados con la query |
| `results[i].score` | **relevancia** (0-1) entre la query y el contenido |
| `results[i].raw_content` | la página entera (solo con `include_raw_content`) |
| `response_time`, `request_id` | tiempo de respuesta e identificador |

Todo esto entra en el `ToolMessage` como `str(dict)`. Veamos cuántos tokens son "útiles" y cuántos "relleno":

In [ ]:
from espadas_langgraph_websearchagent.improvements import format_compact

as_dict_text = str(response)
compact_text = format_compact(response["results"])

print(f"Tokens como str(dict) (lo que envía el agente base): {count_text_tokens(model, as_dict_text):,}")
print(f"Tokens solo título + URL + snippet:                  {count_text_tokens(model, compact_text):,}")

## 3. ¿Cómo se generan los snippets? ¿Son la parte relevante?

Según la documentación de Tavily:
1. Tavily descarga la página y la parte en **trozos** (*chunks*) de unos 500 caracteres (`basic`/`fast`) u 800 (`advanced`).
2. Ordena los trozos según lo relacionados que están con la **query**. Ojo: con la query que escribió el LLM, no con la pregunta del usuario.
3. Devuelve los `chunks_per_source` mejores (3 por defecto), unidos con `" [...] "`.
4. Con `search_depth="ultra-fast"` devuelve un resumen NLP de la página en vez de trozos.

Para comprobar si el snippet es la parte relevante, pedimos también la **página entera** (`include_raw_content`) y comparamos:
- **`en pág.`**: cuántos trozos del snippet encontramos literalmente en la página (comprobamos que salen de ahí),
- **`pos.%`**: dónde está el primer trozo dentro de la página (0% = principio). Si siempre es ~0%, Tavily podría estar cogiendo menús o cabeceras,
- **`clave snip.` / `clave pág.`**: si la respuesta correcta (palabra clave) está en el snippet y/o en la página.

**Caso interesante:** `clave pág. = True` pero `clave snip. = False`. La página tenía la respuesta, pero el snippet no la recogió, así que el LLM no la ve.

In [ ]:
from espadas_langgraph_websearchagent.questions import SNIPPET_CASES

snippet_rows = {}
for query, keywords in SNIPPET_CASES:
    print("=" * 110)
    print(f"QUERY: {query}   | palabras clave: {keywords}")
    rows = ta.analyze_snippets(query, keywords, max_results=5)
    ta.print_snippet_analysis(rows)
    snippet_rows[query] = rows

In [ ]:
# Miramos con detalle un snippet: los trozos que lo forman
query = SNIPPET_CASES[0][0]
first = snippet_rows[query][0]
print("URL:", first["url"], "| score:", first["score"])
for i, chunk in enumerate(ta.split_chunks(first["content"]), start=1):
    print(f"\n--- trozo {i} ---\n{chunk}")

In [ ]:
# Resumen: ¿en cuántos resultados la respuesta estaba en la página pero NO en el snippet?
total, missed = 0, 0
for rows in snippet_rows.values():
    for row in rows:
        if row["clave_en_pagina"] is not None:
            total += 1
            if row["clave_en_pagina"] and not row["clave_en_snippet"]:
                missed += 1
print(f"Resultados con palabra clave en la página pero no en el snippet: {missed} de {total}")

## 4. ¿Cómo se genera el score?

Tavily solo dice que el score *"indica la relevancia entre la query y el contenido"*. La fórmula no es pública (es su modelo de ranking). Hacemos dos experimentos para entenderlo mejor:

**a) Misma página, queries distintas:** si el score de una URL cambia según cómo escribimos la query, el score mide el **parecido query-contenido** y no la calidad de la página.

In [ ]:
score_table = ta.compare_scores_across_queries([
    "última versión de Python",
    "Python latest version",
    "Python 3.15 release date",
    "descargar Python",
])

**b) `basic` vs `advanced`:** ¿cambian los resultados y los scores? ¿Compensa pagar el doble?

In [ ]:
depths = ta.compare_search_depths("última versión de Python", max_results=5)

## 5. ¿El score concuerda con la relevancia real?

Un **LLM juez** puntúa de 0 a 10 cada resultado según si ayuda a responder la **pregunta del usuario**. La respuesta tiene que seguir un esquema Pydantic con `relevancia` y `explicacion`. Después comparamos esa nota con el score de Tavily.

Si el score fuera un buen indicador, los dos números deberían subir y bajar juntos (correlación cercana a 1).

In [ ]:
from espadas_langgraph_websearchagent.questions import QUESTIONS

all_rows = []
for question, should_search in QUESTIONS:
    if not should_search:
        continue
    response = tool.invoke({"query": question})
    if not isinstance(response, dict):
        continue
    rows = ta.score_vs_llm_relevance(question, response["results"])
    all_rows.extend(rows)
    print(f"\n=== {question}")
    for row in rows:
        print(f"  tavily={row['score_tavily']:.2f}  llm={row['relevancia_llm']}  {row['url'][:60]}")
        print(f"      → {row['explicacion']}")

In [ ]:
r = ta.correlation([row["score_tavily"] for row in all_rows], [row["relevancia_llm"] for row in all_rows])
print(f"Correlación score Tavily vs relevancia LLM: {r:.2f}" if r is not None else "No hay datos suficientes")

# ¿Cuántos resultados con score alto son poco relevantes según el juez, y al revés?
high_score_low_rel = [x for x in all_rows if x["score_tavily"] >= 0.7 and (x["relevancia_llm"] or 0) < 0.5]
low_score_high_rel = [x for x in all_rows if x["score_tavily"] < 0.5 and (x["relevancia_llm"] or 0) >= 0.7]
print(f"Score alto (>=0.7) pero poco relevantes (<5/10): {len(high_score_low_rel)} de {len(all_rows)}")
print(f"Score bajo (<0.5) pero muy relevantes (>=7/10):  {len(low_score_high_rel)} de {len(all_rows)}")

## 6. ¿El modelo usa las respuestas o alucina?

Dos comprobaciones:
1. **URLs inventadas** (automática y gratis): ¿el agente cita alguna URL que Tavily **no** devolvió?
2. **Grounding con LLM juez**: separa las afirmaciones de la respuesta en *respaldadas* y *no respaldadas* por los snippets (esquema Pydantic `GroundingCheck`).

In [ ]:
from langchain.messages import HumanMessage

agent = searchagent.create_agent_with_search_tool()

questions_to_check = [
    "¿Cuál es la última versión de Python?",
    "Noticias de hoy en la Comunidad de Madrid",
    "¿Cuántos CV tiene la Honda Rebel? Busca los nuevos modelos",
]

for question in questions_to_check:
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    print("=" * 100)
    print("PREGUNTA:", question)
    print(searchagent.get_final_answer(result)[:1500])

    urls = ta.check_cited_urls(result)
    print("\nURLs citadas:", len(urls["urls_citadas"]), "| inventadas:", urls["urls_inventadas"])

    verdict, judge_usage = ta.check_grounding(result)
    if verdict:
        print("¿Usa los resultados?:", verdict.usa_los_resultados)
        print("NO respaldadas:", verdict.afirmaciones_no_respaldadas)
        print("Explicación:", verdict.explicacion)

In [ ]:
# ¿Y cuando NO busca? Ejemplo: el Mundial de 2026 (posterior a su fecha de corte)
result = agent.invoke({"messages": [HumanMessage(content="¿Quién ganó el Mundial de 2026?")]})
print("¿Buscó?:", searchagent.did_search(result))
print(searchagent.get_final_answer(result))

## 7. Conclusiones del Paso 2

> Completa esta sección con tus resultados. Estas son las ideas que se esperan, comprobadas en el código y en la documentación:

- **Lo que ve el LLM:** el schema de `TavilySearch` tiene 9 parámetros con descripciones muy largas. Junto con el prompt oculto de Anthropic (496 tokens) forman la mayor parte de los ~2.000 tokens fijos de **cada** llamada.
- **Lo que devuelve Tavily:** un JSON con `results` (title, url, content, score) y varios campos vacíos. Como el agente lo mete con `str(dict)`, parte de los tokens del ToolMessage son relleno.
- **Snippets:** son trozos de la página elegidos según la **query** del LLM. Si la query es mala, el snippet también lo será. Rellena aquí cuántas veces la respuesta estaba en la página y no en el snippet.
- **Score:** mide el parecido query-contenido, no si la página responde a la pregunta. Cambia con la redacción de la query y no es comparable entre búsquedas. Rellena aquí la correlación con el juez.
- **¿Usa los resultados?:** rellena aquí las URLs inventadas y las afirmaciones no respaldadas. Cuando **no** busca (Mundial 2026), el riesgo de alucinación es máximo.